In [18]:
import pandas as pd

import lsdb


In [19]:
cat_a = lsdb.open_catalog('tests/data/small_sky_order1_collection', columns=['id', 'ra', 'dec'])

In [20]:
cat_a

,id,ra,dec
npartitions=4,,,
"Order: 1, Pixel: 44",int64[pyarrow],double[pyarrow],double[pyarrow]
"Order: 1, Pixel: 45",...,...,...
"Order: 1, Pixel: 46",...,...,...
"Order: 1, Pixel: 47",...,...,...


In [21]:
def rename_cols(df, names_in, names_out):
    """df = rename_cols(df, ['ra', 'dec'], ['my_ra', 'my_dec'])"""
    for name_in, name_out in zip(names_in, names_out):
        df[name_out] = df[name_in]
    col_names = [col for col in df.columns if col not in names_in]
    return df[col_names]

In [22]:
# Scenario B1: Invalid catalog via map_partitions()

# This creates an invalid catalog: the hc_structure contains the old ra and dec column names,
# but now there are no columns with those names.
#
# This SHOULD fail during map_partitions(), because it corrupts the ra/dec columns, but for now it fails silently.
cat_b = cat_a.map_partitions(rename_cols, ['ra', 'dec'], ['my_ra', 'my_dec'])

# We don't get an error until we try to use the catalog:
try:
    cat_a.crossmatch(cat_b)
    print('fine!')
except Exception as e:
    print(repr(e))

ValueError("right table 'small_sky_order1' must have column 'ra'")


/Users/heather/repos/lsdb/src/lsdb/catalog/catalog.py:410: FutureWarning: The default suffix behavior will change from applying suffixes to all columns to only applying suffixes to overlapping columns in a future release.To maintain the current behavior, explicitly set `suffix_method='all_columns'`. To change to the new behavior, set `suffix_method='overlapping_columns'`.
  warnings.warn(


In [ ]:
# Scenario B2: Invalid catalog via map_partitions()

# Should fail because map_partitions() changes ra/dec columns
try:
    # map_partitions(..., compute_single_partition=True) now errors when ra/dec columns change
    cat_b = cat_a.map_partitions(rename_cols, ['ra', 'dec'], ['my_ra', 'my_dec'], compute_single_partition=True)
    print('fine!')
except Exception as e:
    print(repr(e))

Computing Catalog:   0%|          | 0/1 [00:00<?, ?it/s]

ValueError("'ra' not found in result. map_partitions() must not change names of ra or dec columns 'ra', 'dec'.")


In [ ]:
# Scenario A: Creating an invalid catalog via from_dataframe() / DataFrameCatalogLoader
# missing ra/dec

# Should pass
try:
    cat_b = lsdb.from_dataframe(rename_cols(cat_a.compute(), ['ra', 'dec'], ['my_ra', 'my_dec']), 
        ra_column='my_ra', dec_column='my_dec')
    print('fine!')
except Exception as e:
    print(repr(e))

# Should fail during catalog creation
try:
    cat_b = lsdb.from_dataframe(rename_cols(cat_a.compute(), ['ra', 'dec'], ['blah', 'bloo']))
    print('fine!')
except Exception as e:
    print(repr(e))

Computing Catalog:   0%|          | 0/4 [00:00<?, ?it/s]

fine!


Computing Catalog:   0%|          | 0/4 [00:00<?, ?it/s]

ValueError("No column found for 'ra' (required). You can supply ra/dec column names using the arguments `ra_column`, `dec_column`.")


In [6]:
# Scenario A2: Creating an invalid catalog via from_dataframe() / DataFrameCatalogLoader
# ambiguous column matches

ambiguous_df = cat_a.compute()
ambiguous_df['rA'] = ambiguous_df['ra']

# Should fail: ambiguous columns
try:
    cat_b = lsdb.from_dataframe(ambiguous_df)
    print("fine!")
except Exception as e:
    print(repr(e))


ambiguous_df = cat_a.compute()
ambiguous_df['Dec'] = ambiguous_df['dec']

# Should fail: ambiguous columns
try:
    cat_b = lsdb.from_dataframe(ambiguous_df)
    print("fine!")
except Exception as e:
    print(repr(e))

Computing Catalog:   0%|          | 0/4 [00:00<?, ?it/s]

ValueError("Found 2 possible columns for 'ra': ['ra', 'rA']. Please rename columns to disambiguate.")


Computing Catalog:   0%|          | 0/4 [00:00<?, ?it/s]

ValueError("Found 2 possible columns for 'dec': ['dec', 'Dec']. Please rename columns to disambiguate.")


In [8]:
# Scenario B2: Invalid catalog via map_partitions()

# Should fail because map_partitions() changes ra/dec columns
try:
    # map_partitions(..., compute_single_partition=True) now errors when ra/dec columns change
    cat_b = cat_a.map_partitions(rename_cols, ['ra', 'dec'], ['my_ra', 'my_dec'], compute_single_partition=True)
    print('fine!')
except Exception as e:
    print(repr(e))

Computing Catalog:   0%|          | 0/1 [00:00<?, ?it/s]

ValueError("'ra' not found in result. map_partitions() must not change names of ra or dec columns 'ra', 'dec'.")


In [9]:
# Scenario B3: Invalid catalog via map_partitions()

def my_bad_function(df, col_name):
    df[col_name] = df[col_name] + 1
    return df

# Should fail because map_partitions() changes ra/dec values
for col_name in ['ra', 'dec']:
    try:
        cat_b = cat_a.map_partitions(my_bad_function, col_name, compute_single_partition=True)
        print('fine!')
    except Exception as e:
        print(repr(e))



Computing Catalog:   0%|          | 0/1 [00:00<?, ?it/s]

ValueError("ra/dec values have changed. map_partitions() must not change values of ra or dec columns 'ra', 'dec'.")


Computing Catalog:   0%|          | 0/1 [00:00<?, ?it/s]

ValueError("ra/dec values have changed. map_partitions() must not change values of ra or dec columns 'ra', 'dec'.")


In [10]:
# Scenario C: Alternative known ra/dec names

# Should pass
df = pd.DataFrame({
    'raj2000': [1.0, 2.0, 3.0],
    'dej2000': [4.0, 5.0, 6.0]
})
cat_b = lsdb.from_dataframe(df)

# Should pass
df = pd.DataFrame({
    'raMean': [1.0, 2.0, 3.0],
    'decMean': [4.0, 5.0, 6.0]
})
cat_b = lsdb.from_dataframe(df)

In [11]:
# Scenario D: Alternative known ra/dec names cause ambiguous matches

# Should fail
df = pd.DataFrame({
    'ra': [1.0, 2.0, 3.0],
    'dec': [4.0, 5.0, 6.0],
    'raj2000': [1.0, 2.0, 3.0]
})
try:
    cat_b = lsdb.from_dataframe(df)
    print('fine!')
except Exception as e:
    print(repr(e))


ValueError("Found 2 possible columns for 'ra': ['ra', 'raj2000']. Please rename columns to disambiguate.")


In [12]:
# Scenario E: Heuristic matches and non-matches for novel terms

# should pass with warning
df = pd.DataFrame({
    'ra1234': [1.0, 2.0, 3.0],
    'dec5678': [4.0, 5.0, 6.0]
})
try:
    cat_b = lsdb.from_dataframe(df)
    print('1. fine!')
except Exception as e:
    print(repr(e))

# should fail because these columns should be rejected (therefore no valid ra/dec)
df = pd.DataFrame({
    'ra_err': [1.0, 2.0, 3.0],
    'dec_sig': [4.0, 5.0, 6.0]
})
try:
    cat_b = lsdb.from_dataframe(df)
    print('2. fine!')
except Exception as e:
    print(repr(e))

# should pass because 'ra' should be accepted and 'ra_err' should not
df = pd.DataFrame({
    'ra': [1.0, 2.0, 3.0],
    'ra_err': [1.0, 2.0, 3.0],
    'dec': [4.0, 5.0, 6.0]
})
try:
    cat_b = lsdb.from_dataframe(df)
    print('3. fine!')
except Exception as e:
    print(repr(e))


1. fine!
ValueError("No column found for 'ra' (required). You can supply ra/dec column names using the arguments `ra_column`, `dec_column`.")
3. fine!
